# Lab 4: Multiclass Neural Network from Scratch (MNIST)

**Objective:** Build a fully connected feedforward neural network from scratch (no TensorFlow/PyTorch) to classify MNIST digits (0–9).

- **Architecture:** 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax)
- **Optimizers:** SGD, Momentum, Nesterov, Adagrad, RMSProp, Adam
- **Batching:** Full batch, Stochastic (1 sample), Mini-batch (64)

## 1. Imports and MNIST Data Loading

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Load MNIST (60k train, 10k test). First run may download data.
try:
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
except TypeError:
    mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_all = np.array(mnist.data, dtype=np.float32)
y_all = np.array(mnist.target, dtype=np.int32)

# Train/test split (60k train, 10k test)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=10000, random_state=42, stratify=y_all
)
# Rescale to [0,1] and flatten is already 784
X_train = X_train / 255.0
X_test = X_test / 255.0

# One-hot encode labels (N, 10)
def one_hot(y, num_classes=10):
    n = y.shape[0]
    oh = np.zeros((n, num_classes), dtype=np.float32)
    oh[np.arange(n), y] = 1.0
    return oh

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Labels: 0-9, one-hot shape:", y_train_oh.shape)

Train shape: (60000, 784) Test shape: (10000, 784)
Labels: 0-9, one-hot shape: (60000, 10)


## 2. Neural Network from Scratch (784 → 128 → 64 → 10)

In [2]:
class NeuralNet:
    """Fully connected feedforward: 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax)."""

    def __init__(self, layer_sizes=(784, 128, 64, 10), seed=42):
        np.random.seed(seed)
        self.W = []
        self.b = []
        for i in range(len(layer_sizes) - 1):
            # He init for ReLU layers
            scale = np.sqrt(2.0 / layer_sizes[i])
            self.W.append(np.random.randn(layer_sizes[i], layer_sizes[i + 1]) * scale)
            self.b.append(np.zeros((1, layer_sizes[i + 1])))

    def _relu(self, z):
        return np.maximum(0, z)

    def _relu_derivative(self, z):
        return (z > 0).astype(np.float32)

    def _softmax(self, z):
        e = np.exp(z - np.max(z, axis=1, keepdims=True))
        return e / np.sum(e, axis=1, keepdims=True)

    def forward(self, X, return_cache=True):
        """Forward pass. Returns output and cache for backward."""
        cache = {'A': [X], 'Z': []}
        A = X
        for i in range(len(self.W) - 1):
            Z = A @ self.W[i] + self.b[i]
            cache['Z'].append(Z)
            A = self._relu(Z)
            cache['A'].append(A)
        Z_last = A @ self.W[-1] + self.b[-1]
        cache['Z'].append(Z_last)
        out = self._softmax(Z_last)
        cache['A'].append(out)
        return (out, cache) if return_cache else out

    def backward(self, cache, y_batch):
        """Backward pass: cross-entropy + softmax gives (pred - y)."""
        m = y_batch.shape[0]
        dA = cache['A'][-1] - y_batch  # dL/dZ for softmax+CE
        grads_W = []
        grads_b = []
        for i in range(len(self.W) - 1, -1, -1):
            A_prev = cache['A'][i]
            dW = (A_prev.T @ dA) / m
            db = np.sum(dA, axis=0, keepdims=True) / m
            grads_W.insert(0, dW)
            grads_b.insert(0, db)
            if i > 0:
                dA = (dA @ self.W[i].T) * self._relu_derivative(cache['Z'][i - 1])
        return grads_W, grads_b

    def cross_entropy_loss(self, pred, y):
        eps = 1e-12
        pred = np.clip(pred, eps, 1 - eps)
        return -np.mean(np.sum(y * np.log(pred), axis=1))

    def accuracy(self, X, y_labels):
        pred = self.forward(X, return_cache=False)
        return np.mean(np.argmax(pred, axis=1) == y_labels)

## 3. Optimizers from Scratch (no built-in optimizers)

In [3]:
def optimizer_sgd(nn, grads_W, grads_b, lr, state=None):
    """Vanilla gradient descent."""
    for i in range(len(nn.W)):
        nn.W[i] -= lr * grads_W[i]
        nn.b[i] -= lr * grads_b[i]
    return state

def optimizer_momentum(nn, grads_W, grads_b, lr, state, momentum=0.9):
    """Momentum-based gradient descent."""
    if state is None:
        state = {'v_W': [np.zeros_like(w) for w in nn.W],
                 'v_b': [np.zeros_like(b) for b in nn.b]}
    for i in range(len(nn.W)):
        state['v_W'][i] = momentum * state['v_W'][i] + grads_W[i]
        state['v_b'][i] = momentum * state['v_b'][i] + grads_b[i]
        nn.W[i] -= lr * state['v_W'][i]
        nn.b[i] -= lr * state['v_b'][i]
    return state

def optimizer_nesterov(nn, grads_W, grads_b, lr, state, momentum=0.9):
    """Nesterov accelerated gradient: v_new = mu*v + g; theta -= lr*(mu*v_new + g)."""
    if state is None:
        state = {'v_W': [np.zeros_like(w) for w in nn.W],
                 'v_b': [np.zeros_like(b) for b in nn.b]}
    for i in range(len(nn.W)):
        state['v_W'][i] = momentum * state['v_W'][i] + grads_W[i]
        state['v_b'][i] = momentum * state['v_b'][i] + grads_b[i]
        nn.W[i] -= lr * (momentum * state['v_W'][i] + grads_W[i])
        nn.b[i] -= lr * (momentum * state['v_b'][i] + grads_b[i])
    return state

def optimizer_adagrad(nn, grads_W, grads_b, lr, state, eps=1e-8):
    """Adagrad."""
    if state is None:
        state = {'s_W': [np.zeros_like(w) for w in nn.W],
                 's_b': [np.zeros_like(b) for b in nn.b]}
    for i in range(len(nn.W)):
        state['s_W'][i] += grads_W[i] ** 2
        state['s_b'][i] += grads_b[i] ** 2
        nn.W[i] -= lr * grads_W[i] / (np.sqrt(state['s_W'][i]) + eps)
        nn.b[i] -= lr * grads_b[i] / (np.sqrt(state['s_b'][i]) + eps)
    return state

def optimizer_rmsprop(nn, grads_W, grads_b, lr, state, decay=0.99, eps=1e-8):
    """RMSProp."""
    if state is None:
        state = {'s_W': [np.zeros_like(w) for w in nn.W],
                 's_b': [np.zeros_like(b) for b in nn.b]}
    for i in range(len(nn.W)):
        state['s_W'][i] = decay * state['s_W'][i] + (1 - decay) * (grads_W[i] ** 2)
        state['s_b'][i] = decay * state['s_b'][i] + (1 - decay) * (grads_b[i] ** 2)
        nn.W[i] -= lr * grads_W[i] / (np.sqrt(state['s_W'][i]) + eps)
        nn.b[i] -= lr * grads_b[i] / (np.sqrt(state['s_b'][i]) + eps)
    return state

def optimizer_adam(nn, grads_W, grads_b, lr, state, beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam (time step t stored in state)."""
    if state is None:
        state = {'m_W': [np.zeros_like(w) for w in nn.W], 'm_b': [np.zeros_like(b) for b in nn.b],
                 'v_W': [np.zeros_like(w) for w in nn.W], 'v_b': [np.zeros_like(b) for b in nn.b], 't': 0}
    state['t'] = state.get('t', 0) + 1
    t = state['t']
    for i in range(len(nn.W)):
        state['m_W'][i] = beta1 * state['m_W'][i] + (1 - beta1) * grads_W[i]
        state['m_b'][i] = beta1 * state['m_b'][i] + (1 - beta1) * grads_b[i]
        state['v_W'][i] = beta2 * state['v_W'][i] + (1 - beta2) * (grads_W[i] ** 2)
        state['v_b'][i] = beta2 * state['v_b'][i] + (1 - beta2) * (grads_b[i] ** 2)
        m_hat_W = state['m_W'][i] / (1 - beta1 ** t)
        m_hat_b = state['m_b'][i] / (1 - beta1 ** t)
        v_hat_W = state['v_W'][i] / (1 - beta2 ** t)
        v_hat_b = state['v_b'][i] / (1 - beta2 ** t)
        nn.W[i] -= lr * m_hat_W / (np.sqrt(v_hat_W) + eps)
        nn.b[i] -= lr * m_hat_b / (np.sqrt(v_hat_b) + eps)
    return state

OPTIMIZERS = {
    'SGD': optimizer_sgd,
    'Momentum': optimizer_momentum,
    'Nesterov': optimizer_nesterov,
    'Adagrad': optimizer_adagrad,
    'RMSProp': optimizer_rmsprop,
    'Adam': optimizer_adam,
}

## 4. Training with Batching Strategies

- **Batch GD:** entire dataset per epoch  
- **Stochastic GD:** one sample per update  
- **Mini-batch GD:** batch_size = 64

In [4]:
def train(nn, X_train, y_train_oh, y_train, X_test, y_test,
          optimizer_name='SGD', batch_mode='mini', batch_size=64,
          epochs=10, lr=0.01, seed=42):
    """
    batch_mode: 'full' | 'stochastic' | 'mini'
    Returns: history dict with 'train_loss', 'train_acc', 'test_acc' per epoch.
    """
    opt_fn = OPTIMIZERS[optimizer_name]
    state = None
    n = X_train.shape[0]
    if batch_mode == 'full':
        batch_size = n
    elif batch_mode == 'stochastic':
        batch_size = 1
    # else mini: use batch_size (e.g. 64)

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    for epoch in range(epochs):
        perm = np.random.permutation(n)
        X_shuf = X_train[perm]
        y_shuf_oh = y_train_oh[perm]
        y_shuf = y_train[perm]
        epoch_loss = 0.0
        n_batches = 0
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            X_b = X_shuf[start:end]
            y_b_oh = y_shuf_oh[start:end]
            y_b = y_shuf[start:end]

            pred, cache = nn.forward(X_b, return_cache=True)
            loss = nn.cross_entropy_loss(pred, y_b_oh)
            grads_W, grads_b = nn.backward(cache, y_b_oh)

            kwargs = {}
            if optimizer_name == 'Momentum' or optimizer_name == 'Nesterov':
                kwargs['momentum'] = 0.9
            state = opt_fn(nn, grads_W, grads_b, lr, state, **kwargs)

            epoch_loss += loss
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        train_acc = nn.accuracy(X_train, y_train)
        test_acc = nn.accuracy(X_test, y_test)
        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{epochs}  loss={avg_loss:.4f}  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")
    return history

## 5. Run Experiments (Optimizers & Batching)

Using **mini-batch size 64** and **8 epochs** for quick comparison. Increase `epochs` for better accuracy.

In [5]:
EPOCHS = 8
LR = 0.001
MINI_BATCH = 64

# 5.1 Compare all optimizers (mini-batch 64)
print("=== Comparing optimizers (mini-batch=64) ===\n")
results_optimizers = {}
for name in OPTIMIZERS:
    print(f"--- {name} ---")
    nn = NeuralNet(seed=42)
    hist = train(nn, X_train, y_train_oh, y_train, X_test, y_test,
                 optimizer_name=name, batch_mode='mini', batch_size=MINI_BATCH,
                 epochs=EPOCHS, lr=LR)
    results_optimizers[name] = {'history': hist, 'model': nn}
    print()

=== Comparing optimizers (mini-batch=64) ===

--- SGD ---
  Epoch 1/8  loss=2.0115  train_acc=0.6212  test_acc=0.6221
  Epoch 2/8  loss=1.3180  train_acc=0.7658  test_acc=0.7677
  Epoch 4/8  loss=0.7029  train_acc=0.8395  test_acc=0.8412
  Epoch 6/8  loss=0.5350  train_acc=0.8630  test_acc=0.8628
  Epoch 8/8  loss=0.4594  train_acc=0.8781  test_acc=0.8740

--- Momentum ---
  Epoch 1/8  loss=0.8191  train_acc=0.8843  test_acc=0.8817
  Epoch 2/8  loss=0.3588  train_acc=0.9089  test_acc=0.9057
  Epoch 4/8  loss=0.2628  train_acc=0.9315  test_acc=0.9279
  Epoch 6/8  loss=0.2167  train_acc=0.9419  test_acc=0.9381
  Epoch 8/8  loss=0.1851  train_acc=0.9504  test_acc=0.9485

--- Nesterov ---
  Epoch 1/8  loss=0.8159  train_acc=0.8846  test_acc=0.8827
  Epoch 2/8  loss=0.3585  train_acc=0.9089  test_acc=0.9060
  Epoch 4/8  loss=0.2626  train_acc=0.9316  test_acc=0.9286
  Epoch 6/8  loss=0.2165  train_acc=0.9419  test_acc=0.9383
  Epoch 8/8  loss=0.1849  train_acc=0.9505  test_acc=0.9484

--- A

In [ ]:
# 5.2 Compare batching strategies (using Adam)
print("=== Comparing batching strategies (Adam) ===\n")
results_batching = {}
for mode, bs in [('full', 60000), ('stochastic', 1), ('mini', 64)]:
    print(f"--- {mode} (batch_size={bs}) ---")
    nn = NeuralNet(seed=43)
    hist = train(nn, X_train, y_train_oh, y_train, X_test, y_test,
                 optimizer_name='Adam', batch_mode=mode, batch_size=bs,
                 epochs=EPOCHS, lr=LR)
    results_batching[mode] = {'history': hist, 'model': nn}
    print()

=== Comparing batching strategies (Adam) ===

--- full (batch_size=60000) ---
  Epoch 1/8  loss=2.4064  train_acc=0.1653  test_acc=0.1654


## 6. Visualization & Interpretation

### 6.1 Training loss and accuracy curves (different optimizers)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for name, data in results_optimizers.items():
    h = data['history']
    axes[0].plot(h['train_loss'], label=name)
    axes[1].plot(h['train_acc'], label=name)
    axes[2].plot(h['test_acc'], label=name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].set_title('Loss (Optimizers)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train Accuracy'); axes[1].set_title('Train Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Test Accuracy'); axes[2].set_title('Test Accuracy')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Convergence: Batch vs Stochastic vs Mini-batch (Adam)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for mode, data in results_batching.items():
    h = data['history']
    axes[0].plot(h['train_loss'], label=mode)
    axes[1].plot(h['test_acc'], label=mode)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].set_title('Loss by Batching')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy'); axes[1].set_title('Test Accuracy by Batching')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Confusion matrix (best: Adam + mini-batch)

In [ ]:
# Use best model: Adam with mini-batch (from results_optimizers)
best_nn = results_optimizers['Adam']['model']
y_pred = np.argmax(best_nn.forward(X_test, return_cache=False), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix (Adam, mini-batch=64)')
plt.tight_layout()
plt.show()
print("Test accuracy (best):", np.mean(y_pred == y_test))

### Summary

- **Optimizers:** SGD is slowest; Momentum/Nesterov speed convergence; Adagrad/RMSProp/Adam adapt learning rate and typically give better test accuracy.
- **Batching:** Full-batch is stable but few updates per epoch; stochastic is noisy but many updates; mini-batch (e.g. 64) balances speed and stability.
- **Best setup:** Adam with mini-batch is a strong default; confusion matrix above shows per-class performance.